# Fitur agregasi per grup
**tabular › feature_engineering_regression** · notebook `groupby_aggregates`

**Kapan dipakai:** Rata-rata/median/rank fitur per kategori (mis. harga rata-rata per kelas). Menangkap konteks relatif suatu baris.

**Jebakan umum:** Hitung agregat dari TRAINING saja lalu petakan ke test. Agregat memakai target = kebocoran kecuali di-cross-fit.

## 0. Setup
Import dan lokasi proyek. Tidak perlu diubah.

In [ ]:
import sys, pathlib, warnings
ROOT = next(p for p in [pathlib.Path.cwd().resolve(), *pathlib.Path.cwd().resolve().parents]
            if (p / "src" / "superdb").is_dir())
sys.path.insert(0, str(ROOT / "src"))
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
pd.set_option("display.max_columns", 50)
from superdb.common import *
from superdb.paths import out_dir

## 1. Konfigurasi
Ganti bagian ini saja saat memakai data lomba.

In [ ]:
TRAIN_PATH = ROOT / "data_latihan/tabular/regression/diamonds_train.csv"
TEST_PATH  = ROOT / "data_latihan/tabular/regression/diamonds_test.csv"      # None kalau tidak ada file test
TARGET     = "price"
ID_COL     = 'id'
DROP_COLS  = []
GROUP_COL  = None                      # isi bila ada grup agar tidak bocor
SCORING    = "neg_root_mean_squared_error"
SEED, N_SPLITS, VALID_SIZE = 42, 5, 0.2
MODEL_KEY  = "groupby_aggregates"
TASK       = "regression"
KIND       = "tree"
OUT = out_dir("tabular", "feature_engineering_regression", MODEL_KEY)
set_seed(SEED)

## 2. Baca data

In [ ]:
df = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH) if TEST_PATH else None
print(df.shape, None if test is None else test.shape)
display(df.head(3))

## 3. Fitur & split

In [ ]:
from superdb.tabular import detect_columns, prepare_features, build_pipeline, split_train_valid, cv_score
cols = detect_columns(df, target=TARGET, id_col=ID_COL, drop_cols=DROP_COLS)
X, NUM, CAT = prepare_features(df, cols)
y = df[TARGET].to_numpy(dtype=float)
groups = df[GROUP_COL] if GROUP_COL else None
Xtr, Xva, ytr, yva = split_train_valid(X, y, TASK, VALID_SIZE, SEED, groups)
GTR = None if groups is None else groups.iloc[Xtr.index]   # grup untuk CV
print(Xtr.shape, Xva.shape)

## 4. Cara mengukur: skor CV baseline vs setelah fitur ditambah
Satu model yang sama (LightGBM) dipakai terus; yang berubah hanya fiturnya. Gain di bawah kira-kira `std` = tidak bisa dibedakan dari noise.

In [ ]:
from lightgbm import LGBMRegressor

model = LGBMRegressor(n_estimators=400, learning_rate=0.05, num_leaves=31, min_child_samples=10,
                      subsample=0.8, subsample_freq=1, colsample_bytree=0.8, random_state=42, n_jobs=-1, verbose=-1)
from sklearn.base import clone
def cv_variant(d, label, mdl=None, kind="tree"):
    """CV pada seluruh data train untuk satu varian fitur."""
    cols_ = detect_columns(d, target=TARGET, id_col=ID_COL, drop_cols=DROP_COLS, verbose=False)
    X_, num_, cat_ = prepare_features(d, cols_)
    _, s = cv_score(build_pipeline(clone(mdl or model), num_, cat_, kind=kind), X_, y, TASK, SCORING, N_SPLITS, SEED)
    return {"varian": label, "n_fitur": X_.shape[1], "cv_mean": s["valid_mean"], "cv_std": s["valid_std"]}

## 5. Agregat per grup di dalam pipeline
`GroupAggregator` mempelajari rata-rata grup dari data training saja di setiap fold (tidak bocor), lalu menambah kolom rata-rata grup dan selisih baris terhadap rata-rata grupnya.

In [ ]:
from sklearn.compose import ColumnTransformer, make_column_selector as sel
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder
from superdb.tabular_extra_b import GroupAggregator
Xc = X.copy(); Xc[CAT] = Xc[CAT].astype(str)
pre = ColumnTransformer([
    ("num", SimpleImputer(strategy="median"), sel(dtype_include="number")),
    ("cat", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1), sel(dtype_exclude="number"))])
base_pipe = Pipeline([("pre", pre), ("model", clone(model))])
agg_pipe = Pipeline([("agg", GroupAggregator([["cut"], ["color"], ["clarity"]], ["carat", "depth", "table"])), ("pre", pre), ("model", clone(model))])
rows = []
for name, pp in {"baseline": base_pipe, "+ agregat per grup": agg_pipe}.items():
    _, s = cv_score(pp, Xc, y, TASK, SCORING, N_SPLITS, SEED)
    rows.append({"varian": name, "cv_mean": s["valid_mean"], "cv_std": s["valid_std"]})
res = pd.DataFrame(rows); res["delta_vs_baseline"] = res.cv_mean - res.cv_mean.iloc[0]
display(res.round(4))
display(GroupAggregator([["cut"], ["color"], ["clarity"]], ["carat", "depth", "table"]).fit(Xc).transform(Xc.head(3)).iloc[:, -6:].round(2))

## 6. Gambar perbandingan

In [ ]:
lab = (res.encoding + "/" + res.model) if "encoding" in res.columns else res.iloc[:, 0]
fig, ax = plt.subplots(figsize=(7, max(2.5, .35 * len(res))))
ax.barh(lab, res.cv_mean, xerr=res.cv_std, color="tab:blue", alpha=.8)
lo = (res.cv_mean - res.cv_std).min(); hi = (res.cv_mean + res.cv_std).max(); ax.set_xlim(lo - .1 * abs(hi - lo), hi + .1 * abs(hi - lo))
ax.set_xlabel(f"CV {SCORING} (garis = std antar fold)"); plt.tight_layout(); plt.savefig(OUT / "fe_compare.png", dpi=130); plt.show()
res.to_csv(OUT / "fe_results.csv", index=False)
save_json({"method": MODEL_KEY, "results": res.to_dict("records")}, OUT / "metrics.json")

## Langkah berikutnya
- Fitur yang terbukti naik di CV **dan** masuk akal secara domain dimasukkan ke pipeline akhir.
- Gain kecil dibanding `std`: abaikan, jangan menambah kompleksitas.
- Ukur selalu dengan CV, bukan satu split.